# F2-M06: Análisis de Evolución

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## Qué hace
Análisis de la evolución temporal del dataset: matrículas, notas, nuevos
vs continuos, egresados, becas, ramas, créditos y trayectorias individuales
(spaghetti plot).

## Requisitos
- `df_alumno.parquet` en `data/02_processed/`
- Módulos: `src.config`, `src.utils`, `src.html`

## Genera
- `docs/html/fase2/m06_evolucion.html`
- `docs/html/fase2/graficos/m06_*.html` (9+ gráficos)

## Flujo
```
... → M05 Univariante Cat → **M06 Evolución** → M07 Conclusiones → ...
```

## Siguiente
`f2_m07_conclusiones.ipynb`

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO
# ============================================================================
# - Detecta entorno (Colab / local)
# - Localiza ROOT buscando src/ (robusto, sin hardcodes)
# - Importa módulos del proyecto
# - Crea directorios de salida
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# --- Detectar entorno y localizar ROOT ---
def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError(
        f"No se encontró carpeta 'src/' subiendo desde {start}. "
        f"Verifica que el notebook está dentro de AU_UJI/"
    )

ROOT = _encontrar_root(Path.cwd())

sys.path.insert(0, str(ROOT))

# --- Imports ---
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.config import RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.utils import crear_directorios, formato_numero_es
from src.html import (
    generar_kpis_html,
    generar_seccion_html,
    generar_html_navegacion_completa,
    guardar_html
)
from src.html.render import render_pagina_desde_fichero

# --- Rutas de salida ---
RUTA_FASE2 = RUTA_HTML / 'fase2'
RUTA_GRAFICOS = RUTA_FASE2 / 'graficos'
crear_directorios([RUTA_FASE2, RUTA_GRAFICOS])

info_entorno()

✓ Directorios verificados: 2
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS
# ============================================================================

print('=' * 60)
print('CARGANDO DATOS')
print('=' * 60)

df = pd.read_parquet(RUTA_PROCESSED / 'df_alumno.parquet')

# Identificar columna de curso
curso_col = None
for col in ['curso_aca', 'curso_aca_id', 'curso_academico']:
    if col in df.columns:
        curso_col = col
        break

if curso_col is None:
    raise ValueError('No se encontró columna de curso académico')

print(f'✅ Dataset cargado: {len(df):,} filas × {len(df.columns)} columnas')
print(f'📅 Columna de curso: {curso_col}')

CARGANDO DATOS
✅ Dataset cargado: 109,568 filas × 37 columnas
📅 Columna de curso: curso_aca


In [3]:
# ============================================================================
# CELDA 3: PREPARAR DATOS DE EVOLUCIÓN
# ============================================================================

print('=' * 60)
print('PREPARANDO DATOS DE EVOLUCIÓN')
print('=' * 60)

# Matrículas por curso
matriculas_curso = df.groupby(curso_col).size().sort_index()
cursos = matriculas_curso.index.tolist()
cursos_str = [str(c) for c in cursos]
n_cursos = len(cursos)
curso_min = cursos[0]
curso_max = cursos[-1]

# Alumnos únicos por curso
if 'per_id_ficticio' in df.columns:
    alumnos_curso = df.groupby(curso_col)['per_id_ficticio'].nunique()
else:
    alumnos_curso = matriculas_curso

# Crecimiento
mat_inicial = matriculas_curso.iloc[0]
mat_final = matriculas_curso.iloc[-1]
crecimiento_total = ((mat_final - mat_inicial) / mat_inicial * 100)

print(f'📅 Cursos: {n_cursos} ({curso_min} - {curso_max})')
print(f'📈 Crecimiento total: {crecimiento_total:+.1f}%')

# --- Calcular créditos superados del año y suspendidos ---
if 'cred_superados' in df.columns and 'cred_matriculados' in df.columns:
    df = df.sort_values(['per_id_ficticio', curso_col])
    
    # Superados año anterior del mismo alumno
    df['_sup_anterior'] = df.groupby('per_id_ficticio')['cred_superados'].shift(1)
    
    # Superados del año = diferencia con año anterior
    # Primer año del alumno: superados_anno = cred_superados (todo es nuevo)
    df['cred_superados_anno'] = df['cred_superados'] - df['_sup_anterior']
    df.loc[df['_sup_anterior'].isnull(), 'cred_superados_anno'] = df.loc[df['_sup_anterior'].isnull(), 'cred_superados']
    df['cred_superados_anno'] = df['cred_superados_anno'].clip(lower=0)
    
    # Suspendidos del año = matriculados - superados del año
    df['cred_suspendidos_anno'] = (df['cred_matriculados'] - df['cred_superados_anno']).clip(lower=0)
    
    # Limpiar columna temporal
    df = df.drop('_sup_anterior', axis=1)
    
    print(f'✅ Nuevas columnas calculadas:')
    print(f'   cred_superados_anno: media={df["cred_superados_anno"].mean():.1f}')
    print(f'   cred_suspendidos_anno: media={df["cred_suspendidos_anno"].mean():.1f}')
    



PREPARANDO DATOS DE EVOLUCIÓN
📅 Cursos: 11 (2010 - 2020)
📈 Crecimiento total: +214.8%


✅ Nuevas columnas calculadas:
   cred_superados_anno: media=43.4
   cred_suspendidos_anno: media=16.3


### 📌 Nota de contexto: Cohorte 2019-2020 — Impacto COVID-19

El último curso del dataset (2019-2020) coincide con el inicio de la pandemia de
COVID-19 en España. En marzo de 2020 las universidades suspendieron la actividad
presencial y adaptaron la docencia y la evaluación de forma urgente.

**Efectos esperados en los datos de este curso:**
- Posible reducción de créditos matriculados o variación en tasas de superación
- Variaciones en el número de egresados (convocatorias extraordinarias)
- Alteraciones en patrones de beca y acceso

**Interpretación:** cualquier anomalía observada en la cohorte 2019-2020 debe
interpretarse con cautela — puede reflejar el efecto COVID más que tendencias
estructurales del sistema universitario. Este curso se mantiene en el análisis
pero se señala explícitamente como atípico.


In [4]:
# ============================================================================
# CELDA 4: GRÁFICO 1 - EVOLUCIÓN MATRÍCULAS Y ALUMNOS (COMBINADO)
# ============================================================================

print('=' * 60)
print('GRÁFICO: EVOLUCIÓN MATRÍCULAS Y ALUMNOS')
print('=' * 60)

fig1 = make_subplots(specs=[[{"secondary_y": True}]])

# Barras: Matrículas
fig1.add_trace(
    go.Bar(
        x=cursos_str,
        y=matriculas_curso.values,
        name='Matrículas',
        marker_color='#3182ce',
        opacity=0.8
    ),
    secondary_y=False
)

# Línea: Alumnos únicos
fig1.add_trace(
    go.Scatter(
        x=cursos_str,
        y=alumnos_curso.values,
        name='Alumnos únicos',
        mode='lines+markers',
        line=dict(color='#2d3748', width=2),
        marker=dict(size=8)
    ),
    secondary_y=True
)

fig1.update_layout(
    title='📊 Evolución de Matrículas y Alumnos Únicos',
    height=400,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    margin=dict(t=80, b=50, l=60, r=60)
)
fig1.update_xaxes(title_text='Curso Académico', tickangle=-45)
fig1.update_yaxes(title_text='Matrículas', secondary_y=False)
fig1.update_yaxes(title_text='Alumnos Únicos', secondary_y=True)

fig1.write_html(RUTA_GRAFICOS / 'm06_evolucion_matriculas.html', include_plotlyjs='cdn')
print('✅ Gráfico de matrículas generado')

GRÁFICO: EVOLUCIÓN MATRÍCULAS Y ALUMNOS


✅ Gráfico de matrículas generado

In [5]:
# ============================================================================
# CELDA 5: GRÁFICO 2 - NOTAS CON BANDA DE DESVIACIÓN
# ============================================================================

print('=' * 60)
print('GRÁFICO: EVOLUCIÓN DE NOTAS')
print('=' * 60)

# Buscar columna de notas
nota_col = None
for col in ['nota_media', 'media_titulacion_curso', 'nota_media_curso', 'media']:
    if col in df.columns:
        nota_col = col
        break

if nota_col:
    notas_stats = df.groupby(curso_col)[nota_col].agg(['mean', 'std'])
    nota_global = df[nota_col].mean()
    
    fig2 = go.Figure()
    
    # Banda de desviación estándar
    fig2.add_trace(go.Scatter(
        x=cursos_str + cursos_str[::-1],
        y=list(notas_stats['mean'] + notas_stats['std']) + list((notas_stats['mean'] - notas_stats['std'])[::-1]),
        fill='toself',
        fillcolor='rgba(49, 130, 206, 0.2)',
        line=dict(color='rgba(255,255,255,0)'),
        name='± Desv. Estándar',
        showlegend=True
    ))
    
    # Línea de media
    fig2.add_trace(go.Scatter(
        x=cursos_str,
        y=notas_stats['mean'],
        mode='lines+markers',
        name='Nota Media',
        line=dict(color='#3182ce', width=3),
        marker=dict(size=8)
    ))
    
    # Línea de media global
    fig2.add_hline(
        y=nota_global, 
        line_dash='dash', 
        line_color='#718096',
        annotation_text=f'Media global: {nota_global:.2f}',
        annotation_position='right'
    )
    
    fig2.update_layout(
        title='📚 Evolución de Notas Medias por Curso',
        xaxis_title='Curso Académico', xaxis_tickangle=-45,
        yaxis_title='Nota Media',
        height=400,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        margin=dict(t=80, b=50, l=60, r=60)
    )
    
    fig2.write_html(RUTA_GRAFICOS / 'm06_evolucion_notas.html', include_plotlyjs='cdn')
    print(f'✅ Gráfico de notas generado (media global: {nota_global:.2f})')
else:
    nota_global = 0
    print('⚠️ No se encontró columna de notas')

GRÁFICO: EVOLUCIÓN DE NOTAS
✅ Gráfico de notas generado (media global: 6.07)


In [6]:
# ============================================================================
# CELDA 6: GRÁFICO 3 - NUEVOS VS CONTINUOS (STACKED + LÍNEA %)
# ============================================================================

print('=' * 60)
print('GRÁFICO: ALUMNOS NUEVOS VS CONTINUOS')
print('=' * 60)

tiene_nuevos = 'nuevo' in df.columns

if tiene_nuevos:
    nuevos_curso = df.groupby([curso_col, 'nuevo']).size().unstack(fill_value=0)
    
    # Calcular porcentaje de nuevos
    if 'S' in nuevos_curso.columns:
        pct_nuevos = (nuevos_curso['S'] / nuevos_curso.sum(axis=1) * 100)
        n_nuevos = nuevos_curso['S'].values if 'S' in nuevos_curso.columns else [0]*len(cursos)
        n_continuos = nuevos_curso['N'].values if 'N' in nuevos_curso.columns else [0]*len(cursos)
    else:
        pct_nuevos = pd.Series([0]*len(cursos), index=cursos)
        n_nuevos = [0]*len(cursos)
        n_continuos = [0]*len(cursos)
    
    # Gráfico stacked
    fig3 = make_subplots(specs=[[{"secondary_y": True}]])
    
    fig3.add_trace(
        go.Bar(x=cursos_str, y=n_nuevos, name='Nuevos', marker_color='#38a169'),
        secondary_y=False
    )
    fig3.add_trace(
        go.Bar(x=cursos_str, y=n_continuos, name='Continuos', marker_color='#63b3ed'),
        secondary_y=False
    )
    
    fig3.update_layout(
        title='👥 Alumnos Nuevos vs Continuos por Curso',
        barmode='stack',
        height=400,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        margin=dict(t=80, b=50, l=60, r=60)
    )
    fig3.update_xaxes(title_text='Curso Académico', tickangle=-45)
    fig3.update_yaxes(title_text='Número de Matrículas', secondary_y=False)
    
    fig3.write_html(RUTA_GRAFICOS / 'm06_nuevos_continuos.html', include_plotlyjs='cdn')
    
    # Gráfico % nuevos con área
    fig3b = go.Figure()
    fig3b.add_trace(go.Scatter(
        x=cursos_str,
        y=pct_nuevos.values,
        mode='lines+markers',
        fill='tozeroy',
        fillcolor='rgba(56, 161, 105, 0.3)',
        line=dict(color='#38a169', width=2),
        marker=dict(size=8),
        name='% Nuevos'
    ))
    fig3b.update_layout(
        title='📊 Porcentaje de Alumnos Nuevos por Curso',
        xaxis_title='Curso Académico',
        yaxis_title='% Alumnos Nuevos',
        height=350,
        margin=dict(t=60, b=50, l=60, r=40)
    )
    fig3b.update_xaxes(tickangle=-45)
    fig3b.write_html(RUTA_GRAFICOS / 'm06_pct_nuevos.html', include_plotlyjs='cdn')
    
    print('✅ Gráficos de nuevos/continuos generados')
else:
    print('⚠️ Columna "nuevo" no encontrada')

GRÁFICO: ALUMNOS NUEVOS VS CONTINUOS
✅ Gráficos de nuevos/continuos generados


In [7]:
# ============================================================================
# CELDA 7: GRÁFICO 4 - EGRESADOS (BARRAS + TASA)
# ============================================================================

print('=' * 60)
print('GRÁFICO: EGRESADOS')
print('=' * 60)

tiene_egresado = 'egresado' in df.columns

if tiene_egresado:
    egresados_curso = df[df['egresado'] == 'S'].groupby(curso_col).size()
    # Rellenar cursos sin egresados
    egresados_curso = egresados_curso.reindex(cursos, fill_value=0)
    
    # Tasa de egreso
    tasa_egreso = (egresados_curso / matriculas_curso * 100)
    
    fig4 = make_subplots(specs=[[{"secondary_y": True}]])
    
    fig4.add_trace(
        go.Bar(
            x=cursos_str,
            y=egresados_curso.values,
            name='Egresados',
            marker_color='#38a169'
        ),
        secondary_y=False
    )
    
    fig4.add_trace(
        go.Scatter(
            x=cursos_str,
            y=tasa_egreso.values,
            name='Tasa Egreso (%)',
            mode='lines+markers',
            line=dict(color='#2d3748', width=2),
            marker=dict(size=8)
        ),
        secondary_y=True
    )
    
    fig4.update_layout(
        title='🎓 Egresados por Curso Académico',
        height=400,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        margin=dict(t=80, b=50, l=60, r=60)
    )
    fig4.update_xaxes(title_text='Curso Académico', tickangle=-45)
    fig4.update_yaxes(title_text='Egresados', secondary_y=False)
    fig4.update_yaxes(title_text='Tasa Egreso (%)', secondary_y=True)
    
    fig4.write_html(RUTA_GRAFICOS / 'm06_egresados.html', include_plotlyjs='cdn')
    print('✅ Gráfico de egresados generado')
else:
    print('⚠️ Columna "egresado" no encontrada')

GRÁFICO: EGRESADOS
✅ Gráfico de egresados generado


In [8]:
# ============================================================================
# CELDA 8: GRÁFICO 5 - EVOLUCIÓN POR RAMA (ÁREA STACKED)
# ============================================================================

print('=' * 60)
print('GRÁFICO: EVOLUCIÓN POR RAMA')
print('=' * 60)

tiene_rama = 'rama' in df.columns

if tiene_rama:
    rama_curso = df.groupby([curso_col, 'rama']).size().unstack(fill_value=0)
    
    # Colores para ramas
    colores_rama = ['#3182ce', '#e53e3e', '#38a169', '#805ad5', '#ed8936']
    
    # Gráfico de área stacked
    fig5 = go.Figure()
    
    for i, rama in enumerate(rama_curso.columns):
        fig5.add_trace(go.Scatter(
            x=cursos_str,
            y=rama_curso[rama].values,
            name=rama,
            mode='lines',
            stackgroup='one',
            line=dict(width=0.5, color=colores_rama[i % len(colores_rama)]),
            fillcolor=colores_rama[i % len(colores_rama)]
        ))
    
    fig5.update_layout(
        title='📚 Evolución de Matrículas por Rama de Conocimiento',
        xaxis_title='Curso Académico',
        yaxis_title='Matrículas',
        height=400,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        margin=dict(t=80, b=50, l=60, r=40)
    )
    
    fig5.update_xaxes(tickangle=-45)
    fig5.write_html(RUTA_GRAFICOS / 'm06_rama_evolucion.html', include_plotlyjs='cdn')
    
    # Heatmap — escala logarítmica para que ramas pequeñas se vean
    import math
    rama_curso_log = rama_curso.applymap(lambda x: math.log10(x + 1) if x > 0 else 0)
    import plotly.express as px
    fig5b = px.imshow(
        rama_curso.T,
        title='Heatmap: Matrículas por Rama y Curso<br>'
              '<span style="font-size:11px; color:#718096;">'
              'Normalizado por rama (color = % del máximo de cada rama)</span>',
        color_continuous_scale='Blues',
        aspect='auto',
        labels=dict(x='Curso', y='Rama', color='Matrículas')
    )
    fig5b.update_layout(height=350, margin=dict(t=60, b=50, l=60, r=40))
    fig5b.update_xaxes(tickangle=-45, dtick=1)
    fig5b.write_html(RUTA_GRAFICOS / 'm06_heatmap_rama.html', include_plotlyjs='cdn')
    
    print('✅ Gráficos de rama generados')
else:
    print('⚠️ Columna "rama" no encontrada')

GRÁFICO: EVOLUCIÓN POR RAMA


✅ Gráficos de rama generados


In [9]:
# ============================================================================
# CELDA 9: GRÁFICO 6 - BECAS (BARRAS + % LÍNEA)
# ============================================================================

print('=' * 60)
print('GRÁFICO: EVOLUCIÓN DE BECAS')
print('=' * 60)

tiene_beca = 'tiene_beca' in df.columns or 'nombre_beca' in df.columns

if tiene_beca:
    if 'tiene_beca' in df.columns:
        becas_count = df.groupby(curso_col)['tiene_beca'].sum()
        becas_pct = df.groupby(curso_col)['tiene_beca'].mean() * 100
    else:
        becas_count = df.groupby(curso_col)['nombre_beca'].apply(lambda x: x.notna().sum())
        becas_pct = df.groupby(curso_col)['nombre_beca'].apply(lambda x: x.notna().mean()) * 100
    
    fig6 = make_subplots(specs=[[{"secondary_y": True}]])
    
    fig6.add_trace(
        go.Bar(
            x=cursos_str,
            y=becas_count.values,
            name='Con beca',
            marker_color='#63b3ed'
        ),
        secondary_y=False
    )
    
    fig6.add_trace(
        go.Scatter(
            x=cursos_str,
            y=becas_pct.values,
            name='% beca',
            mode='lines+markers',
            line=dict(color='#2d3748', width=2),
            marker=dict(size=8)
        ),
        secondary_y=True
    )
    
    fig6.update_layout(
        title='💰 Evolución de Becas',
        height=400,
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        margin=dict(t=80, b=50, l=60, r=60)
    )
    fig6.update_xaxes(title_text='Curso Académico', tickangle=-45)
    fig6.update_yaxes(title_text='Nº con Beca', secondary_y=False)
    fig6.update_yaxes(title_text='% con Beca', secondary_y=True)
    
    fig6.write_html(RUTA_GRAFICOS / 'm06_becas.html', include_plotlyjs='cdn')
    print('✅ Gráfico de becas generado')
else:
    print('⚠️ Columna de becas no encontrada')

GRÁFICO: EVOLUCIÓN DE BECAS
✅ Gráfico de becas generado


In [10]:
# ============================================================================
# CELDA 10: GRÁFICO 7 - CRÉDITOS (5 LÍNEAS)
# ============================================================================
# Usa las columnas calculadas en celda 3:
#   cred_matriculados: dato original (del año)
#   cred_superados: acumulado (dato original)
#   cred_superados_anno: calculado (superados ese año)
#   cred_suspendidos_anno: calculado (matriculados - superados del año)
# ============================================================================

print('=' * 60)
print('GRÁFICO: CRÉDITOS')
print('=' * 60)

tiene_creditos = 'cred_superados_anno' in df.columns

if tiene_creditos:
    por_curso = df.groupby(curso_col).agg(
        mat=('cred_matriculados', 'sum'),
        sup_acum=('cred_superados', 'sum'),
        sup_anno=('cred_superados_anno', 'sum'),
        susp_anno=('cred_suspendidos_anno', 'sum')
    ).sort_index()
    
    miles = por_curso / 1000
    tasa_exito = (por_curso['sup_anno'] / por_curso['mat'] * 100).round(1)
    cursos_str = por_curso.index.astype(str)

    fig7 = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Matriculados del año (azul)
    fig7.add_trace(go.Scatter(
        x=cursos_str, y=miles['mat'].values,
        name='Matriculados (año)', mode='lines+markers',
        line=dict(color='#3182ce', width=2.5),
        marker=dict(size=8),
        hovertemplate='Curso %{x}<br>Matriculados: %{y:,.0f}k<extra></extra>'
    ), secondary_y=False)
    
    # Superados del año (verde)
    fig7.add_trace(go.Scatter(
        x=cursos_str, y=miles['sup_anno'].values,
        name='Superados (año)', mode='lines+markers',
        line=dict(color='#38a169', width=2.5),
        marker=dict(size=8),
        hovertemplate='Curso %{x}<br>Superados año: %{y:,.0f}k<extra></extra>'
    ), secondary_y=False)
    
    # Suspendidos del año (rojo)
    fig7.add_trace(go.Scatter(
        x=cursos_str, y=miles['susp_anno'].values,
        name='Suspendidos (año)', mode='lines+markers',
        line=dict(color='#e53e3e', width=2),
        marker=dict(size=7),
        hovertemplate='Curso %{x}<br>Suspendidos año: %{y:,.0f}k<extra></extra>'
    ), secondary_y=False)
    
    # Superados acumulados (gris punteado, referencia)
    fig7.add_trace(go.Scatter(
        x=cursos_str, y=miles['sup_acum'].values,
        name='Superados (acumulado)', mode='lines',
        line=dict(color='#a0aec0', width=1.5, dash='dot'),
        hovertemplate='Curso %{x}<br>Acumulado: %{y:,.0f}k<extra></extra>'
    ), secondary_y=False)
    
    # Tasa éxito (eje derecho)
    fig7.add_trace(go.Scatter(
        x=cursos_str, y=tasa_exito.values,
        name='Tasa Éxito (%)', mode='lines+markers',
        line=dict(color='#2d3748', width=2, dash='dash'),
        marker=dict(size=7, symbol='diamond'),
        hovertemplate='Curso %{x}<br>Tasa éxito: %{y:.1f}%<extra></extra>'
    ), secondary_y=True)
    
    fig7.update_layout(
        title=None,
        height=480,
        legend=dict(orientation='h', yanchor='bottom', y=1.04, xanchor='center', x=0.5),
        margin=dict(t=60, b=80, l=60, r=60)
    )
    fig7.update_xaxes(title_text='Curso Académico', tickangle=-45)
    fig7.update_yaxes(title_text='Créditos (miles)', secondary_y=False)
    fig7.update_yaxes(title_text='Tasa Éxito (%)', secondary_y=True)
    
    fig7.write_html(RUTA_GRAFICOS / 'm06_creditos.html', include_plotlyjs='cdn')
    print(f'✅ Gráfico de créditos: 5 líneas')
    print(f'   Tasa éxito media: {tasa_exito.mean():.1f}%')
else:
    print('⚠️ Ejecutar celda 3 primero (calcula cred_superados_anno)')


GRÁFICO: CRÉDITOS
✅ Gráfico de créditos: 5 líneas
   Tasa éxito media: 76.3%


In [11]:
# ============================================================================
# CELDA NEW: GRÁFICO — SPAGHETTI PLOT POR TITULACIÓN
# ============================================================================
# Selecciona 4 titulaciones representativas (una por rama grande)
# Cada titulación un color, egresados línea continua, no egresados punteada
# ============================================================================

nota_tray_col = None
for col in ['media_curso', 'media_titulacion_curso', 'nota']:
    if col in df.columns:
        nota_tray_col = col
        break

if nota_tray_col and 'per_id_ficticio' in df.columns and 'titulacion' in df.columns:
    cursos_ordenados = sorted(df[curso_col].unique())
    cursos_str_ord = [str(c) for c in cursos_ordenados]
    
    # Top 4 titulaciones por número de alumnos (diversas)
    top_titulaciones = df.groupby('titulacion')['per_id_ficticio'].nunique().sort_values(ascending=False)
    
    # Seleccionar 4 de diferentes ramas si es posible
    seleccionadas = []
    ramas_vistas = set()
    for tit in top_titulaciones.index:
        if len(seleccionadas) >= 4:
            break
        rama_tit = df[df['titulacion'] == tit]['rama'].mode()
        rama_val = rama_tit.iloc[0] if len(rama_tit) > 0 else 'OTRA'
        if rama_val not in ramas_vistas:
            seleccionadas.append(tit)
            ramas_vistas.add(rama_val)
    
    # Si no llegamos a 4 con ramas diferentes, completar con las más grandes
    if len(seleccionadas) < 4:
        for tit in top_titulaciones.index:
            if tit not in seleccionadas:
                seleccionadas.append(tit)
            if len(seleccionadas) >= 4:
                break
    
    colores_tit = ['#3182ce', '#e53e3e', '#38a169', '#ed8936']
    
    fig_spag = go.Figure()
    
    for idx_t, tit in enumerate(seleccionadas):
        color = colores_tit[idx_t]
        df_tit = df[df['titulacion'] == tit].copy()
        
        # Alumnos con ≥3 cursos en esta titulación
        cursos_por_al = df_tit.groupby('per_id_ficticio')[curso_col].nunique()
        al_3plus = cursos_por_al[cursos_por_al >= 3].index
        
        if len(al_3plus) == 0:
            continue
        
        # Máximo 8 alumnos por titulación
        n_muestra = min(8, len(al_3plus))
        muestra = np.random.RandomState(42 + idx_t).choice(al_3plus, n_muestra, replace=False)
        
        # Nombre abreviado para leyenda
        nombre_corto = str(tit).replace('Grado en ', 'G. ').replace('Grau en ', 'G. ')
        if len(nombre_corto) > 30:
            nombre_corto = nombre_corto[:28] + '..'
        
        primero_egr = True
        primero_no = True
        
        for alumno in muestra:
            datos_a = df_tit[df_tit['per_id_ficticio'] == alumno].sort_values(curso_col)
            notas = datos_a[nota_tray_col].values
            cursos_a = datos_a[curso_col].astype(str).values
            
            if len(notas[~np.isnan(notas)]) < 2:
                continue
            
            es_egresado = (datos_a['egresado'] == 'S').any() if 'egresado' in datos_a.columns else False
            dash_style = 'solid' if es_egresado else 'dash'
            
            # Leyenda solo para el primero de cada tipo
            if es_egresado and primero_egr:
                show_leg = True
                nombre_leg = f'{nombre_corto} (egresado)'
                primero_egr = False
            elif not es_egresado and primero_no:
                show_leg = True
                nombre_leg = f'{nombre_corto} (no egr.)'
                primero_no = False
            else:
                show_leg = False
                nombre_leg = ''
            
            fig_spag.add_trace(go.Scatter(
                x=cursos_a, y=notas,
                mode='lines+markers',
                line=dict(color=color, width=1.8, dash=dash_style),
                marker=dict(size=4),
                name=nombre_leg,
                showlegend=show_leg,
                legendgroup=f'{tit}_{es_egresado}',
                hovertemplate=f'{nombre_corto}<br>'
                              f'Alumno: {alumno}<br>'
                              f'{"✅ Egresado" if es_egresado else "❌ No egresado"}<br>'
                              f'Curso: %{{x}}<br>Nota: %{{y:.2f}}<extra></extra>'
            ))
    
    # Media global (negro grueso)
    media_por_curso = df.groupby(curso_col)[nota_tray_col].agg(['mean', 'std']).reindex(cursos_ordenados)
    
    # Banda ±1σ
    fig_spag.add_trace(go.Scatter(
        x=cursos_str_ord + cursos_str_ord[::-1],
        y=(media_por_curso['mean'] + media_por_curso['std']).tolist() + 
           (media_por_curso['mean'] - media_por_curso['std']).tolist()[::-1],
        fill='toself', fillcolor='rgba(113,128,150,0.12)',
        line=dict(color='rgba(0,0,0,0)'),
        showlegend=False, hoverinfo='skip'
    ))
    
    fig_spag.add_trace(go.Scatter(
        x=cursos_str_ord, y=media_por_curso['mean'],
        mode='lines+markers',
        line=dict(color='#2D3748', width=3),
        marker=dict(size=7),
        name='Media global',
        hovertemplate='Curso: %{x}<br>Media: %{y:.2f}<extra></extra>'
    ))

    fig_spag.update_layout(
        title=None,
        xaxis_title='Curso Académico',
        yaxis_title='Nota Media del Curso',
        xaxis=dict(categoryorder='array', categoryarray=cursos_str_ord, tickangle=-45),
        height=500,
        margin=dict(t=30, b=70, l=60, r=220),
        legend=dict(font=dict(size=9), yanchor='top', y=0.98, xanchor='left', x=1.02,
                    orientation='v', bgcolor='rgba(255,255,255,0.8)')
    )
    fig_spag.write_html(RUTA_GRAFICOS / 'm06_spaghetti.html', include_plotlyjs='cdn')
    tiene_spaghetti = True
    print(f'✅ Spaghetti: {len(seleccionadas)} titulaciones')
    for t in seleccionadas:
        print(f'   • {t}')
else:
    tiene_spaghetti = False
    print('⚠️ No se genera spaghetti')


✅ Spaghetti: 4 titulaciones
   • Grado en Administración de Empresas
   • Grado en Psicología
   • Grado en Ingeniería en Diseño Industrial y Desarrollo de Productos
   • Grado en Traducción e Interpretación


In [12]:
# ============================================================================
# CELDA: CONSTRUIR KPIs Y SECCIONES HTML
# ============================================================================

# --- KPIs ---
KPIS = [
    {'valor': str(n_cursos), 'titulo': 'Cursos Académicos'},
    {'valor': f'{curso_min}-{curso_max}', 'titulo': 'Período'},
    {'valor': f'{crecimiento_total:+.1f}%', 'titulo': 'Crecimiento Total'},
    {'valor': f'{nota_global:.2f}' if nota_col else 'N/D', 'titulo': 'Nota Media Global'},
]
kpis_html = generar_kpis_html(KPIS)

# --- Sección 1: Matrículas y Notas ---
seccion_mat_notas = generar_seccion_html(
    titulo="Evolución de Matrículas y Notas", icono="📈",
    contenido='''<div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
        <iframe src="graficos/m06_evolucion_matriculas.html" width="100%" height="420" frameborder="0"></iframe>
        <iframe src="graficos/m06_evolucion_notas.html" width="100%" height="420" frameborder="0"></iframe>
    </div>'''
) if nota_col else generar_seccion_html(
    titulo="Evolución de Matrículas", icono="📈",
    contenido='<iframe src="graficos/m06_evolucion_matriculas.html" width="100%" height="420" frameborder="0"></iframe>'
)

# --- Sección 2: Nuevos vs Continuos ---
seccion_nuevos = generar_seccion_html(
    titulo="Alumnos Nuevos vs Continuos", icono="👥",
    contenido='''<div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
        <iframe src="graficos/m06_nuevos_continuos.html" width="100%" height="420" frameborder="0"></iframe>
        <iframe src="graficos/m06_pct_nuevos.html" width="100%" height="420" frameborder="0"></iframe>
    </div>'''
) if tiene_nuevos else ''

# --- Sección 3: Egresados y Becas ---
if tiene_egresado and tiene_beca:
    seccion_egr_becas = generar_seccion_html(
        titulo="Egresados y Becas", icono="🎓",
        contenido='''<div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
            <iframe src="graficos/m06_egresados.html" width="100%" height="420" frameborder="0"></iframe>
            <iframe src="graficos/m06_becas.html" width="100%" height="420" frameborder="0"></iframe>
        </div>'''
    )
elif tiene_egresado:
    seccion_egr_becas = generar_seccion_html(
        titulo="Egresados por Curso", icono="🎓",
        contenido='<iframe src="graficos/m06_egresados.html" width="100%" height="420" frameborder="0"></iframe>'
    )
elif tiene_beca:
    seccion_egr_becas = generar_seccion_html(
        titulo="Evolución de Becas", icono="💰",
        contenido='<iframe src="graficos/m06_becas.html" width="100%" height="420" frameborder="0"></iframe>'
    )
else:
    seccion_egr_becas = ''

# --- Sección 4: Rama ---
seccion_rama = generar_seccion_html(
    titulo="Evolución por Rama de Conocimiento", icono="📚",
    contenido='''<div style="display:grid; grid-template-columns:1fr 1fr; gap:20px;">
        <iframe src="graficos/m06_rama_evolucion.html" width="100%" height="420" frameborder="0"></iframe>
        <iframe src="graficos/m06_heatmap_rama.html" width="100%" height="420" frameborder="0"></iframe>
    </div>'''
) if tiene_rama else ''

# --- Sección 5: Créditos ---
seccion_cred = generar_seccion_html(
    titulo="Créditos Matriculados vs Superados", icono="📊",
    contenido='<iframe src="graficos/m06_creditos.html" width="100%" height="420" frameborder="0"></iframe>'
) if tiene_creditos else ''

# --- Sección 6: Trayectorias ---
seccion_tray = generar_seccion_html(
    titulo="Trayectorias Individuales", icono="🕸️",
    contenido='<iframe src="graficos/m06_spaghetti.html" width="100%" height="470" frameborder="0"></iframe>'
) if tiene_spaghetti else ''

print('✅ Secciones HTML construidas')


✅ Secciones HTML construidas


In [13]:
# ============================================================================
# CELDA 11: GENERAR HTML (2 GRÁFICOS POR FILA)
# ============================================================================

print("=" * 60)
print("GENERANDO HTML")
print("=" * 60)

# --- Ensamblar contenido ---
contenido_html = (kpis_html + seccion_mat_notas + seccion_nuevos +
                  seccion_egr_becas + seccion_rama + seccion_cred + seccion_tray)

# --- Generar HTML ---
html_completo = render_pagina_desde_fichero(
    'f2_m06_evolucion.ipynb',
    contenido_html,
    carpeta_notebook='fase2_eda'
)
ruta_html = RUTA_FASE2 / "m06_evolucion.html"
ruta_html.parent.mkdir(parents=True, exist_ok=True)
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'✅ HTML generado: {ruta_html}')

print(f"\n✅ HTML generado: {ruta_html}")

GENERANDO HTML
✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m06_evolucion.html

✅ HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m06_evolucion.html


In [14]:
# ============================================================================
# CELDA 12: RESUMEN
# ============================================================================

print('\n' + '=' * 60)
print('✅ F2-M06 COMPLETADO')
print('=' * 60)
print(f'\n📁 HTML: {ruta_html}')
print(f'📊 Gráficos generados:')
print(f'   - m06_evolucion_matriculas.html (barras + línea)')
print(f'   - m06_evolucion_notas.html (línea + banda desviación)')
print(f'   - m06_nuevos_continuos.html (stacked bars)')
print(f'   - m06_pct_nuevos.html (área)')
print(f'   - m06_egresados.html (barras + tasa)')
print(f'   - m06_becas.html (barras + %)')
print(f'   - m06_rama_evolucion.html (área stacked)')
print(f'   - m06_heatmap_rama.html')
print(f'   - m06_creditos.html (líneas + tasa éxito)')
print(f'\n📌 Siguiente: f2_m07_conclusiones.ipynb')
print('=' * 60)


✅ F2-M06 COMPLETADO

📁 HTML: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase2\m06_evolucion.html
📊 Gráficos generados:
   - m06_evolucion_matriculas.html (barras + línea)
   - m06_evolucion_notas.html (línea + banda desviación)
   - m06_nuevos_continuos.html (stacked bars)
   - m06_pct_nuevos.html (área)
   - m06_egresados.html (barras + tasa)
   - m06_becas.html (barras + %)
   - m06_rama_evolucion.html (área stacked)
   - m06_heatmap_rama.html
   - m06_creditos.html (líneas + tasa éxito)

📌 Siguiente: f2_m07_conclusiones.ipynb
